## Proceso de resolución

---------------------


No mires esto hasta haberte partido los cuernos tú solito.

Está claro que deberemos iterar por todos los elementos de la matriz (lista de listas).

### Índices

Para acceder a cada elemento, usaremos dos índices (i y j), siendo que i representa la columna y j la fila.

|        |        |        |        |
|--------|--------|--------|--------|
| (0,0)  | (0,1)  | (0,2)  | (0,3)  |
| (1,0)  | (1,1)  | (1,2)  | (1,3)  |
| (2,0)  | (2,1)  | (2,2)  | (2,3)  |
| (3,0)  | (3,1)  | (3,2)  | (3,3)  |


Para recorrer todos los elementos, vamos a necesitar un bucle anidado (un for dentro del otro):

```python
for i, column in enumerate(matrix):
        for j, value in enumerate(column):
                new_value = process_element(....)
```

### `enumerate`

La función `enumerate` en Python es útil para obtener tanto el índice como el valor de los elementos de un iterable dentro de un bucle `for`. Esto simplifica el proceso de contar iteraciones, ya que `enumerate` automáticamente devuelve un contador junto con el valor del elemento del iterable.

#### Sintaxis básica:
```python
enumerate(iterable, start=0)
```
- `iterable`: Cualquier objeto iterable sobre el que se pueda iterar, como listas, cadenas, tuplas, etc.
- `start`: El valor inicial del contador, por defecto es 0.

##### Ejemplo de uso:

Supongamos que tienes una lista de frutas y quieres imprimir cada fruta con su índice correspondiente:

```python
frutas = ['manzana', 'banana', 'cereza', 'durazno']

for indice, fruta in enumerate(frutas):
    print(indice, fruta)
```

### Salida:
```
0 manzana
1 banana
2 cereza
3 durazno
```

Cuando veamos las tuplas, entenderemos mejor lo que hace `enumerate`.


### `process_element`

Vista la estructura general, procesar una lista y devolver otra, cosa que ya hemos visto mil veces, podemos centrarnos en la transformación que vamos a aplicar a cada uno de los elementos.

La función `process_element` va a recibir los índices del elemento y la matriz original, y devolverá el nuevo valor.

### Divide y Vencerás

A primera vista, la tarea se puede descomponer en las siguientes subtareas:

1. Descubrir los vecinos de un elemento
2. Obtener su promedio
3. Promediarlo con el valor antiguo

El principal escollo parece ser el hecho de que según cual sea el elemento, **el número de vecinos cambia**. Da la impresión de que tendremos tres reglas de vecindario: 

* interior, 
* borde y 
* esquina.


### Versión Reducida

Para poder empezar, vamos a resolver una versión reducida del problema, antes de ponernos con la versión completa. En vez de una matriz, vamos a **procesar una lista**.

La función `average_list` hace lo mismo que la función `average_matrix`, pero actúa sobre una lista de números.

Esto simplifica el bucle principal y sobre todo simplifica la tarea de encontrar vecinos. Ahora solo hay dos tipos (en vez de tres):


* **Interno**: tiene dos vecinos
* **Borde (inicial o final)**: el primero y el último sólo tienen 1 vecino



|INICIAL|INTERNO|INTERNO|INTERNO|INTERNO|FINAL
|--------|--------|--------|--------|-------|-------|
| 0  | 1  | 2  | 3  |   4    |5


Una vez tengas resuelta la versión reducida, puedes ya abordar la versión completa.




### Obtener los vecinos

Esta parte parece un poco rollo, más que nada porque hay dos casos (en la versión reducida de una sola dimensión):

* punto interior: tiene dos vecinos, el de delante y el de atrás
* punto extremo (inicial o final): sólo tiene un vecino (o bien el de antes o el de atrás)

A mí me encantaría que sólo hubiese un casos, el general, y que pudiese olvidarme de los casos excepcionales.  Esto simplificaría enormemente el
código y elimina un montón de posibilidades de errores.

Hay varias estratégias para hacer esto, veamos dos.


#### Los centinelas

Es una técnica clásica. Se trata de insertar un valor *inofensivo* antes del primer elemento y después del último. Sabemos de su existencia y no los tenemos en cuenta,
pero de un plumazo hemos logrado que el primer elemento de verdad, tenga un vecino a la izquierda (el segundo) y a la derecha (el centinela). De igual manera, el último tendrá un vecino a la izquierda (el penúltimo) y también a la derecha (el centinela).

Ahora todos los elementos de verdad, **son internos**.


|CENTINELA|INTERNO|INTERNO|INTERNO|INTERNO|INTERNO|INTERNO|CENTINELA|
|--------|--------|--------|--------|--------|--------|-------|-------|
| None| 0  | 1  | 2  | 3  |   4    |5|None|



A la hora de iterar, tendremos que tenerlo en cuenta y sólo iterar por los internos, sin procesar los centinelas, o volvemos a lo de antes. También habrá que elegir muy vien el valor que metemos en los centinelas, para que alteren el promedio, o bien habrá que identificarlos y no tenerlos en cuenta (por ejemplo, si contiene `None` no lo tengo en cuenta en el promedio).


```python
# Iterar desde el segundo elemento hasta el penúltimo
for elemento in lista[1:-1]:
    # haz algo
```

#### Cagarla y luego arreglarla

Hay otra técnica que nos puede ser más útil. A veces, *resulta más fácil hacer una tarea mal  arreglar los errores a posteriori, que hacerlo bien a la primera. Por ejemplo, si tienes qu epintar una pared.

Si es la primera vez que lo haces, es posible que procedas de la siguiente manera:

1. Pintas todo con un rodillo, procurando mantenerte alejado de interruptores y puertas
2. Gastas horas en pintar alrededor de la spuertas y los interruptores, tomando mucho cuidado para no pintarlos

Un pintor más experimentado sabe que hay una forma mejor:

1. Protege lo qu eno debe de ser pintado con cinta
2. Pinta por encima de todo sin proecuparte
3. Quita la cinta de protección

A veces, *es mejor cagarla y arreglar después*.

**¿Cómo podemos aplicar esto?**

1. Considera que todos los puntos son internos (dos vecinos)
2. Obtén una lista con las coordenadas de los vecinos de cada punto (en el caso de una dimensión, serán dos vecinos y en el caso de la tabla, cuatro)
3. Si es fácil eliminar de esas listas las coordenadas erroneas, tarea resuelta.


Supongamos que tienes un lista de 4 valores (del 0 al 3) y quieres crear una lista de los vecinos de cada elemento y los tratas como si todos fuesen internos. Terminarías con las siguientes listas

0. (**-1**,1)
1. (0,2)
2. (1,3)
3. (2,**4**)

Está claro los qu están mal y que tenemos que quitar, los que tienen índices imposibles: **menores que 0** y **mayores que la longitud de la lista**.

Por lo tanto, generaríamos primero una lista en bruto, y luego seleccionamos sólo los buenos (quitamos la cinta que protegía los interruptores.)


## Resumen

A veces, cuando el problema es muy complejo y no hay forma de reducirlo, es bueno inventar una versión con menos dimensiones para que sea intrínsecamente más sencillo. Al resolver ése, aprenderemos lo bastante como para abordar el completo.

La reducción de dimensiones es común en IA.

#### Mis amigos los centinelas


![](centinelas.png)


Hace muchos años, cuando MS dominaba la tierra yo era un programador aspirante, me sentía muy inseguro. Creía que sabía mucho menso de lo que realmente sabía y estaba dominado por el 
famoso "síndrome del impostor". Un buen día, vi un anuncio de MS donde publicaban un trozo de código de MS Word con una sóla frase: *si ves el bug, queremos conocerte*.

Era un código en C++ que iteraba por una lista: **vi el error instantaneamente**.  Estaban tratando todos los elementos de la lista como si fuesen internos sin tener en cuenta que el primero no tiene a nadie detrás y el último no tiene a nadie delante. 

En C++ eso haría que la aplicación se cayese instantaenamente. **¡Se les olvidó usar centinelas!**

No mandé mi cv por idiota, pero supe que ya era programador. Ese momento también te llegará, joven padawan.